# Minggu 11 — Praktik: Anatomi Seismogram dan Pemrosesan Sinyal

**Seismologi PAGF262413** · Program Studi Sarjana Geofisika FMIPA UGM

Data: **enam gempa susulan Yogyakarta 2006**, stasiun TF14 jaringan YK, 100 Hz, tiga komponen. Rekaman nyata — bukan sintetik, bukan contoh dari buku teks.

---

### Aturan main

**AI boleh dipakai sebebasnya** untuk menulis kode, menjelaskan konsep, dan mencari kesalahan. Wajib dicatat di sel terakhir: apa yang kalian tanyakan dan bagian mana yang berasal dari sana.

Yang dinilai **bukan kodenya**, melainkan tiga hal:

| Yang dinilai | Bagaimana diukur |
|:--|:--|
| Ketepatan pick | \|Δt\| terhadap pick rujukan, dalam detik |
| Ketepatan pilihan pita lewat | Dibandingkan dengan spektrum event kalian sendiri |
| Mutu penalaran | Jawaban tertulis di sel bertanda ✍️ |

Pick kalian akan diadu dengan pick rujukan. **Perhatikan:** pick rujukan itu keluaran EqTransformer, bukan analis manusia — jadi bukan kebenaran mutlak. Kalau kalian yakin pick kalian lebih baik, tuliskan alasannya; itu bernilai.

Posisi P **berbeda-beda di tiap event** dan tidak ada polanya. Jangan mencari pola, carilah gelombangnya.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from obspy import read
plt.rcParams['figure.figsize']=(13,4); plt.rcParams['axes.grid']=True; plt.rcParams['grid.alpha']=.25

st  = read('data/TF14_contoh.mseed')
ref = pd.read_csv('data/TF14_contoh_pick_rujukan.csv')
print(st.__str__(extended=True))
ref

## Bagian 1 — Pilih event kalian

Ganti `EVENT` dengan nomor yang diberikan dosen. Kalau belum diberi, pakai `1` untuk berlatih.

In [ ]:
EVENT = 1                       # <-- ganti sesuai nomor kalian
NAMA  = "tulis nama kalian"     # <-- wajib diisi

sel = st.select(location=f"{EVENT:02d}").copy()
inf = ref[ref.id == f"EV{EVENT:02d}"].iloc[0]
fs  = sel[0].stats.sampling_rate
print(f"{NAMA} | EV{EVENT:02d} | M{inf.M} | jarak {inf.dist_km} km | fs = {fs} Hz")
print(f"panjang {sel[0].stats.npts/fs:.0f} detik, {len(sel)} komponen")

## Bagian 2 — Anatomi tiga komponen

Plot ketiga komponen. **Jangan ditapis dulu.**

✍️ Setelah memplot, jawab di sel berikutnya:
1. Komponen mana yang amplitudo puncaknya paling besar? Fase apa yang menyebabkannya?
2. Bisakah kalian menunjuk P dari data mentah ini? Kalau tidak, mengapa?

In [ ]:
t = np.arange(sel[0].stats.npts)/fs
fig, ax = plt.subplots(3,1, figsize=(13,6), sharex=True)
for i, tr in enumerate(sel):
    ax[i].plot(t, tr.data, lw=.5)
    ax[i].set_ylabel(tr.stats.channel)
ax[2].set_xlabel('Waktu sejak awal rekaman (detik)')
plt.tight_layout()

# TUGAS: hitung amplitudo puncak tiap komponen
for tr in sel:
    print(tr.stats.channel, '-> puncak', ...)      # <-- lengkapi

✍️ **Jawaban 2.1 dan 2.2:**

*(tulis di sini)*

## Bagian 3 — Lihat spektrumnya SEBELUM memilih penapis

Ini urutan yang benar. Memilih pita lewat tanpa melihat spektrum adalah menebak.

In [ ]:
def spektrum(x, fsr):
    x = (x - x.mean()) * np.hanning(len(x))
    return np.fft.rfftfreq(len(x), 1/fsr), np.abs(np.fft.rfft(x))

z = sel.select(component='Z')[0].data.astype(float)

n0, n1 = 0, int(8*fs)                  # jendela derau: 8 detik pertama
s0, s1 = int(...*fs), int(...*fs)      # <-- TUGAS: jendela yang berisi gempa

fN, AN = spektrum(z[n0:n1], fs)
fS, AS = spektrum(z[s0:s1], fs)
plt.loglog(fN[1:], AN[1:], label='derau', color='gray')
plt.loglog(fS[1:], AS[1:], label='sinyal')
plt.xlabel('Frekuensi (Hz)'); plt.ylabel('Amplitudo spektral'); plt.legend()
plt.title('Di pita mana sinyal menang atas derau?')

✍️ **Jawaban 3:** Di rentang frekuensi mana sinyal paling jauh di atas derau?

Pita lewat pilihan kalian: **____ – ____ Hz**. Alasannya:

*(tulis di sini — jawaban tanpa alasan bernilai nol)*

## Bagian 4 — Tapis, lalu pick

Ingat kaidah dari kuliah: untuk **mem-_pick_ waktu tiba** gunakan penapis **kausal** (`zerophase=False`). Penapis zero-phase menimbulkan dering sebelum onset sejati dan membuat kalian mem-_pick_ terlalu awal.

In [ ]:
LO, HI = ..., ...                 # <-- pita pilihan kalian dari Bagian 3

kerja = sel.copy()
kerja.detrend('demean')           # selalu demean/detrend SEBELUM menapis
kerja.filter('bandpass', freqmin=LO, freqmax=HI, corners=4, zerophase=False)

zf = kerja.select(component='Z')[0].data
plt.figure(figsize=(13,3.5)); plt.plot(t, zf, lw=.6)
plt.xlabel('Waktu (detik)'); plt.title(f'Komponen Z — bandpass {LO}-{HI} Hz, kausal')

P_saya = ...       # <-- waktu tiba P, detik sejak awal rekaman
S_saya = ...       # <-- waktu tiba S
print(f'S-P = {S_saya - P_saya:.2f} s')

## Bagian 5 — SEL BERGALAT ⚠️

Sel di bawah ditulis oleh AI dan **tampak masuk akal**. Di dalamnya ada **tiga kesalahan**.

Temukan ketiganya, perbaiki, dan jelaskan akibat masing-masing terhadap hasil.

Melaporkan kesalahan yang sebenarnya tidak ada **mengurangi** nilai — jadi jangan asal menuduh semua baris.

In [ ]:
# ============ SEL BERGALAT — jangan dipakai sebelum diperbaiki ============
tr = sel.select(component='Z')[0].copy()

# (a) turunkan laju cuplik dari 100 Hz ke 25 Hz supaya pemrosesan lebih ringan
data_25 = tr.data[::4]

# (b) tapis pita lebar supaya semua fase tertangkap
tr.filter('bandpass', freqmin=1, freqmax=60, corners=4, zerophase=True)

# (c) hitung amplitudo puncak untuk keperluan magnitudo
amplitudo_puncak = np.max(tr.data)

print('laju cuplik baru :', 25, 'Hz')
print('amplitudo puncak :', amplitudo_puncak)
# =========================================================================

✍️ **Jawaban 5:**

| # | Baris | Kesalahannya | Akibatnya pada hasil |
|:--|:--|:--|:--|
| 1 | | | |
| 2 | | | |
| 3 | | | |

Tulis versi perbaikannya di sel berikut.

In [ ]:
# Versi perbaikan kalian


## Bagian 6 — Aliasing, dibuktikan sendiri

Buktikan bahwa mengambil tiap sampel ke-N tanpa menapis lebih dulu merusak data.

In [ ]:
a = sel.select(component='Z')[0].copy(); a.detrend('demean')
salah = a.data[::4].astype(float)             # cara yang salah
benar = a.copy(); benar.decimate(4)           # cara yang benar (menapis dulu)

f1, A1 = spektrum(salah, fs/4)
f2, A2 = spektrum(benar.data.astype(float), fs/4)
plt.loglog(f1[1:], A1[1:], label='tiap sampel ke-4, tanpa tapis')
plt.loglog(f2[1:], A2[1:], label='decimate() — menapis dulu')
plt.xlabel('Frekuensi (Hz)'); plt.ylabel('Amplitudo spektral'); plt.legend()
plt.title('Akibat aliasing pada spektrum')

✍️ **Jawaban 6:** Berapa Nyquist setelah desimasi ke 25 Hz? Di bagian spektrum mana kedua kurva mulai berbeda, dan mengapa justru di sana?

*(tulis di sini)*

## Bagian 7 — Setoran

In [ ]:
NIM = 'isi NIM kalian'          # <-- wajib, dipakai pengoreksi otomatis

setoran = dict(
    nim=NIM, nama=NAMA, soal=f"W11-{EVENT:02d}",
    P_detik=float(P_saya), S_detik=float(S_saya), SP_detik=float(S_saya - P_saya),
    pita_lo=float(LO), pita_hi=float(HI),
    P_selang_bawah=...,        # <-- batas bawah selang keyakinan 80% untuk pick P
    P_selang_atas=...,         # <-- batas atas. JANGAN dikosongkan.
)
pd.DataFrame([setoran]).to_csv(f'setoran_{NIM}_W11.csv', index=False)
setoran

✍️ **Catatan pemakaian AI** (wajib diisi, tidak dinilai benar-salah):

- Apa saja yang kalian tanyakan ke AI?
- Bagian mana dari notebook ini yang berasal dari sana?
- Bagian mana yang akhirnya kalian ubah sendiri karena jawabannya kurang tepat?

*(tulis di sini)*

---

### Pengingat tentang ketidakpastian

Sel setoran meminta **selang keyakinan** untuk pick P kalian, bukan satu angka. Selang sempit yang meleset lebih buruk daripada selang lebar yang tepat. Melaporkan `± 0,01 s` pada data ber-SNR rendah bukan tanda ketelitian — itu tanda belum paham seberapa besar yang belum kalian ketahui.